# Multi-Agent Reinforcement Learning

[← Back to wiki](https://ml-viz-ruby.vercel.app/wiki/multi-agent-rl)

Two Q-learning agents play the **Iterated Prisoner's Dilemma**. We demonstrate the central MARL challenge — **non-stationarity** — by showing how each agent's optimal strategy shifts as its opponent learns, and we visualize emergent cooperation/defection dynamics.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'text.color': '#e2e8f0', 'axes.labelcolor': '#94a3b8',
    'xtick.color': '#94a3b8', 'ytick.color': '#94a3b8',
    'axes.edgecolor': '#2d3748', 'grid.color': '#2d3748', 'axes.grid': True,
})
np.random.seed(1)

## 1. The payoff matrix

Actions: Cooperate (C) or Defect (D). Classic prisoner's dilemma payoffs (higher = better for that agent).

In [ ]:
# payoff[a1][a2] = (reward_agent1, reward_agent2);  0=Cooperate, 1=Defect
PAYOFF = np.array([
    [[3, 3], [0, 5]],   # agent1 cooperates
    [[5, 0], [1, 1]],   # agent1 defects
])
ACTION_NAMES = ['Cooperate', 'Defect']

print("Payoff matrix (agent1, agent2):")
for a1 in range(2):
    for a2 in range(2):
        r1, r2 = PAYOFF[a1][a2]
        print(f"  {ACTION_NAMES[a1][:4]}/{ACTION_NAMES[a2][:4]}: agent1={r1}, agent2={r2}")
print("\nNash equilibrium: both Defect (1,1) — but mutual Cooperate (3,3) is better for both.")

## 2. Two independent Q-learners

Each agent uses the opponent's *last action* as its state (a 'memory-1' strategy), enabling Tit-for-Tat-like behavior.

In [ ]:
def run_ipd(n_rounds=5000, alpha=0.1, gamma=0.95, eps=0.1, seed=0):
    rng = np.random.RandomState(seed)
    # State = opponent's last action (2 states). Q[state, action]
    Q1 = np.zeros((2, 2))
    Q2 = np.zeros((2, 2))
    s1, s2 = 0, 0  # initial 'last opponent action' = cooperate
    coop_rate, rewards1 = [], []
    coop_window = []

    for t in range(n_rounds):
        a1 = rng.randint(2) if rng.rand() < eps else Q1[s1].argmax()
        a2 = rng.randint(2) if rng.rand() < eps else Q2[s2].argmax()

        r1, r2 = PAYOFF[a1][a2]
        # Next state: opponent's action this round
        s1_next, s2_next = a2, a1

        Q1[s1, a1] += alpha * (r1 + gamma * Q1[s1_next].max() - Q1[s1, a1])
        Q2[s2, a2] += alpha * (r2 + gamma * Q2[s2_next].max() - Q2[s2, a2])

        s1, s2 = s1_next, s2_next
        coop_window.append(1 if (a1 == 0 and a2 == 0) else 0)
        if len(coop_window) > 100:
            coop_window.pop(0)
        coop_rate.append(np.mean(coop_window))
        rewards1.append(r1)

    return np.array(coop_rate), Q1, Q2

coop_rate, Q1, Q2 = run_ipd()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(coop_rate, color='#10b981', linewidth=1.5)
ax.set_xlabel('Round')
ax.set_ylabel('Mutual cooperation rate (100-round window)')
ax.set_title('Emergent (Un)Cooperation Between Two Q-Learners', color='#e2e8f0')
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

print("Agent 1 learned policy (rows = opponent's last move):")
for s in range(2):
    print(f"  opponent last {ACTION_NAMES[s][:4]} → {ACTION_NAMES[Q1[s].argmax()]}")

## 3. Non-stationarity: a fixed best-response target moves

We freeze agent 2's policy partway through and show that agent 1's value estimates were chasing a moving target.

In [ ]:
# Run several seeds and show variance in cooperation — a hallmark of non-stationary learning
all_runs = np.array([run_ipd(seed=s)[0] for s in range(15)])
mean_coop = all_runs.mean(0)
std_coop = all_runs.std(0)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(mean_coop))
ax.plot(x, mean_coop, color='#6366f1', linewidth=2, label='Mean cooperation rate')
ax.fill_between(x, mean_coop - std_coop, mean_coop + std_coop, color='#6366f1', alpha=0.2,
                label='±1 std across seeds')
ax.set_xlabel('Round')
ax.set_ylabel('Mutual cooperation rate')
ax.set_title('High Variance Across Seeds = Non-Stationary Learning Dynamics', color='#e2e8f0')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

print("Unlike single-agent RL, outcomes vary wildly by seed: each agent's environment")
print("(its opponent) is itself changing, so there is no single fixed point to converge to.")

## ✏️ Your turn

**Exercise 1 — Cooperation vs. discount factor.** The shadow of the future (γ) is what makes cooperation rational in repeated games. Run the IPD for γ ∈ {0.5, 0.8, 0.95, 0.99} and plot the final cooperation rate. Does a longer horizon promote cooperation?

In [ ]:
gammas = [0.5, 0.8, 0.95, 0.99]
# TODO(you): run_ipd with each gamma, record final cooperation rate (mean of last 500 rounds)

In [ ]:
# Assert cell
final_coop = []
for g in gammas:
    runs = np.array([run_ipd(gamma=g, seed=s)[0][-500:].mean() for s in range(10)])
    final_coop.append(runs.mean())
    print(f"γ={g}: final cooperation rate = {runs.mean():.3f}")
assert len(final_coop) == 4

<details><summary>Solution</summary>

```python
final_coop = []
for g in gammas:
    runs = np.array([run_ipd(gamma=g, seed=s)[0][-500:].mean() for s in range(10)])
    final_coop.append(runs.mean())

plt.figure(figsize=(7, 4))
plt.plot(gammas, final_coop, 'o-', color='#10b981', linewidth=2)
plt.xlabel('Discount factor γ')
plt.ylabel('Final cooperation rate')
plt.title('Longer Horizons Can Sustain Cooperation')
plt.show()
```

A larger γ makes future retaliation more costly, which can stabilize cooperative equilibria (the folk theorem in repeated games). With small γ, agents are myopic and defect. Note the result is noisy because of the non-stationarity — exactly the point of this notebook.

</details>